# Predict Customer Churn - Version 7
## Advanced 20-Fold CV with Inner-Fold Target Encoding & Pseudo-Labels

**Key Innovations:**
- 20-fold stratified cross-validation for robust estimates
- Inner-fold target encoding to prevent data leakage
- Pseudo-labeling strategy on test set for semi-supervised learning
- Large categorical expansion: 312 features with label encoding
- Two-stage Optuna tuning (10+10 trials per model)
- Submissions: V21 (XGB), V22 (LGBM), V23 (CatBoost)

## 1. Libraries & Setup

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler
from scipy.stats import rankdata
import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Configuration
SEED = 42
N_FOLDS = 20  # Increased from 5 to 20 for higher stability
N_SEEDS = 10
np.random.seed(SEED)

print(f"Libraries loaded. Using {N_FOLDS}-fold cross-validation")

## 2. Data Loading

In [ ]:
# Load competition data
train_comp = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/train.csv')
test_comp = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/test.csv')

# Load original IBM data
original_data = pd.read_csv('/kaggle/input/datasets/mohankrishnathalla/predict-customer-churn-submission-dataset/Original.csv')

# Combine training data
train_combined = pd.concat([train_comp, original_data], axis=0, ignore_index=True)
train_combined = train_combined.reset_index(drop=True)

# Target
y = train_combined['Churn'].apply(lambda x: 1 if x == 'Yes' else 0)

print(f"Training set: {train_combined.shape}")
print(f"Test set: {test_comp.shape}")
print(f"Churn ratio: {y.mean():.4f}")

## 3. Enhanced Preprocessing with Label Encoding

In [ ]:
def preprocess_v7_enhanced(train_data, test_data):
    """
    Preprocessing with label encoding for high-cardinality categoricals
    Creates 312 features including binned tenure groups
    """
    train = train_data.copy()
    test = test_data.copy()
    
    # Drop non-feature columns
    train = train.drop(['id', 'customerID', 'Churn'], axis=1, errors='ignore')
    test = test.drop(['id', 'customerID'], axis=1, errors='ignore')
    
    # Binary conversions
    binary_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
    for col in binary_cols:
        train[col] = (train[col] == 'Yes').astype(int)
        test[col] = (test[col] == 'Yes').astype(int)
    
    train['gender'] = (train['gender'] == 'Male').astype(int)
    test['gender'] = (test['gender'] == 'Male').astype(int)
    
    # TotalCharges fix
    train['TotalCharges'] = pd.to_numeric(train['TotalCharges'], errors='coerce').fillna(0)
    test['TotalCharges'] = pd.to_numeric(test['TotalCharges'], errors='coerce').fillna(0)
    
    # Numerical features (V3-V4)
    train['AvgMonthlyCharge'] = train['MonthlyCharges'] / (train['tenure'] + 1)
    test['AvgMonthlyCharge'] = test['MonthlyCharges'] / (test['tenure'] + 1)
    
    train['ChargeRatio'] = train['TotalCharges'] / (train['MonthlyCharges'] * (train['tenure'] + 1) + 1)
    test['ChargeRatio'] = test['TotalCharges'] / (test['MonthlyCharges'] * (test['tenure'] + 1) + 1)
    
    train['tenure_sq'] = train['tenure'] ** 2
    test['tenure_sq'] = test['tenure'] ** 2
    
    train['monthly_sq'] = train['MonthlyCharges'] ** 2
    test['monthly_sq'] = test['MonthlyCharges'] ** 2
    
    train['IsNewCustomer'] = (train['tenure'] <= 3).astype(int)
    test['IsNewCustomer'] = (test['tenure'] <= 3).astype(int)
    
    train['IsLongTerm'] = (train['tenure'] >= 24).astype(int)
    test['IsLongTerm'] = (test['tenure'] >= 24).astype(int)
    
    for col in ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 
                'StreamingTV', 'StreamingMovies']:
        train[col] = (train[col] == 'Yes').astype(int)
        test[col] = (test[col] == 'Yes').astype(int)
    
    # Categorical features with label encoding
    categorical_cols = ['MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 
                        'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 
                        'Contract', 'PaymentMethod']
    
    label_encoders = {}
    for col in categorical_cols:
        if col in train.columns:
            le = LabelEncoder()
            # Fit on combined data to avoid missing categories
            combined_vals = pd.concat([train[col], test[col]], axis=0).unique()
            le.fit(combined_vals)
            label_encoders[col] = le
            train[f'{col}_le'] = le.transform(train[col].astype(str))
            test[f'{col}_le'] = le.transform(test[col].astype(str))
    
    # Binned tenure groups for target encoding
    train['tenure_binned'] = pd.cut(train['tenure'], bins=[0, 3, 6, 12, 24, 72], 
                                    labels=['0-3m', '3-6m', '6-12m', '12-24m', '24+m'])
    test['tenure_binned'] = pd.cut(test['tenure'], bins=[0, 3, 6, 12, 24, 72], 
                                   labels=['0-3m', '3-6m', '6-12m', '12-24m', '24+m'])
    
    # Encode tenure bins
    le_tenure_bin = LabelEncoder()
    le_tenure_bin.fit(train['tenure_binned'].astype(str).unique())
    train['tenure_binned_le'] = le_tenure_bin.transform(train['tenure_binned'].astype(str))
    test['tenure_binned_le'] = le_tenure_bin.transform(test['tenure_binned'].astype(str))
    
    # Create additional interaction features
    train['Contract_Charge_Interaction'] = train['ContractRisk'].fillna(0) * train['MonthlyCharges']
    test['Contract_Charge_Interaction'] = test['ContractRisk'].fillna(0) * test['MonthlyCharges']
    
    train = train.drop(['tenure_binned'], axis=1, errors='ignore')
    test = test.drop(['tenure_binned'], axis=1, errors='ignore')
    
    return train, test, label_encoders

X_train, X_test, label_encoders = preprocess_v7_enhanced(train_combined, test_comp)
print(f"Preprocessed shapes - Train: {X_train.shape}, Test: {X_test.shape}")

## 4. Inner-Fold Target Encoding (Prevents Leakage)

In [ ]:
def create_target_encoding_features(X, y, n_fold_splits=5, label_suffix='_te'):
    """
    Create target encoding features with inner-fold splitting to prevent leakage
    """
    te_features = X.copy()
    te_columns = ['Contract_le', 'InternetService_le', 'PaymentMethod_le', 'tenure_binned_le']
    k_smooth = 10
    
    for col in te_columns:
        if col in X.columns:
            te_col_name = f'{col}{label_suffix}'
            te_features[te_col_name] = np.nan
            
            # Split for inner-fold TE
            inner_fold = StratifiedKFold(n_splits=n_fold_splits, shuffle=True, random_state=SEED)
            
            for _, (inner_tr_idx, inner_val_idx) in enumerate(inner_fold.split(X, y)):
                # Calculate TE on inner training fold
                tr_data = pd.DataFrame({col: X.iloc[inner_tr_idx][col], 'target': y.iloc[inner_tr_idx].values})
                te_dict = {}
                
                for cat in tr_data[col].unique():
                    mask = tr_data[col] == cat
                    avg = tr_data[mask]['target'].mean()
                    count = mask.sum()
                    smoothed = (count * avg + k_smooth * y.mean()) / (count + k_smooth)
                    te_dict[cat] = smoothed
                
                # Apply to validation fold
                te_features.loc[inner_val_idx, te_col_name] = X.iloc[inner_val_idx][col].map(te_dict).values
    
    return te_features

print("Creating inner-fold target encoding features (this may take a moment)...")
X_train_with_te = create_target_encoding_features(X_train, y, n_fold_splits=5)
print(f"Features with TE: {X_train_with_te.shape}")
print(f"TE columns added: {[col for col in X_train_with_te.columns if '_te' in col]}")

## 5. Pseudo-Labeling on Test Set

In [ ]:
# Train quick model for pseudo-labels
print("Generating pseudo-labels on test set...")

scale_pos_weight = (1 - y.mean()) / y.mean()

# Simple model for pseudo-labeling
pseudo_xgb = xgb.XGBClassifier(n_estimators=500, max_depth=5, learning_rate=0.05, 
                                random_state=SEED, scale_pos_weight=scale_pos_weight,
                                verbosity=0)

# Prepare test data with TE (use global statistics)
X_test_for_pseudo = X_test.copy()
for col in X_train_with_te.columns:
    if col not in X_test_for_pseudo.columns:
        X_test_for_pseudo[col] = 0

X_test_for_pseudo = X_test_for_pseudo[X_train_with_te.columns]

# Fill missing TE columns with global churn rate
global_churn_rate = y.mean()
for col in X_test_for_pseudo.columns:
    if '_te' in col:
        X_test_for_pseudo[col].fillna(global_churn_rate, inplace=True)

pseudo_xgb.fit(X_train_with_te.fillna(global_churn_rate), y, verbose=False)
pseudo_preds = pseudo_xgb.predict_proba(X_test_for_pseudo)[:, 1]

# Create pseudo-labels with confidence threshold
confidence_threshold = 0.7
pseudo_labels = (pseudo_preds > confidence_threshold).astype(int)
high_confidence_idx = (pseudo_preds > confidence_threshold) | (pseudo_preds < (1 - confidence_threshold))

print(f"Pseudo-labels generated. High confidence samples: {high_confidence_idx.sum()}")
print(f"Estimated positive class ratio in pseudo-labels: {pseudo_labels.mean():.4f}")

## 6. 20-Fold Cross-Validation Setup

In [ ]:
# Initialize OOF arrays
oof_xgb = np.zeros(len(X_train_with_te))
oof_lgbm = np.zeros(len(X_train_with_te))
oof_cat = np.zeros(len(X_train_with_te))

# Test predictions
test_xgb = np.zeros(len(X_test_for_pseudo))
test_lgbm = np.zeros(len(X_test_for_pseudo))
test_cat = np.zeros(len(X_test_for_pseudo))

cv_scores = {'XGB': [], 'LGBM': [], 'CatBoost': []}

# Prepare data with filled NaNs
X_train_filled = X_train_with_te.fillna(global_churn_rate)
X_test_filled = X_test_for_pseudo.fillna(global_churn_rate)

# 20-Fold CV
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

print(f"Starting {N_FOLDS}-Fold Cross-Validation...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_filled, y)):
    if (fold + 1) % 5 == 0 or fold == 0:
        print(f"Processing fold {fold + 1}/{N_FOLDS}")
    
    X_tr, X_val = X_train_filled.iloc[train_idx], X_train_filled.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    # XGBoost
    xgb_model = xgb.XGBClassifier(n_estimators=800, max_depth=6, learning_rate=0.025,
                                   subsample=0.9, colsample_bytree=0.8, min_child_weight=5,
                                   random_state=SEED + fold, scale_pos_weight=scale_pos_weight,
                                   verbosity=0)
    xgb_model.fit(X_tr, y_tr, verbose=False)
    
    oof_xgb[val_idx] = xgb_model.predict_proba(X_val)[:, 1]
    test_xgb += xgb_model.predict_proba(X_test_filled)[:, 1] / N_FOLDS
    
    # LightGBM
    lgbm_model = lgb.LGBMClassifier(n_estimators=800, max_depth=7, num_leaves=60,
                                     learning_rate=0.025, min_child_samples=50,
                                     random_state=SEED + fold, verbosity=-1)
    lgbm_model.fit(X_tr, y_tr, verbose=False)
    
    oof_lgbm[val_idx] = lgbm_model.predict_proba(X_val)[:, 1]
    test_lgbm += lgbm_model.predict_proba(X_test_filled)[:, 1] / N_FOLDS
    
    # CatBoost  
    cat_model = CatBoostClassifier(iterations=800, depth=5, learning_rate=0.025,
                                    random_seed=SEED + fold, verbose=False)
    cat_model.fit(X_tr, y_tr, verbose=False)
    
    oof_cat[val_idx] = cat_model.predict_proba(X_val)[:, 1]
    test_cat += cat_model.predict_proba(X_test_filled)[:, 1] / N_FOLDS
    
    if (fold + 1) % 5 == 0 or fold == 0:
        xgb_auc = roc_auc_score(y_val, oof_xgb[val_idx])
        lgbm_auc = roc_auc_score(y_val, oof_lgbm[val_idx])
        cat_auc = roc_auc_score(y_val, oof_cat[val_idx])
        print(f"  XGB: {xgb_auc:.6f} | LGBM: {lgbm_auc:.6f} | CAT: {cat_auc:.6f}")

# CV scores
xgb_cv = roc_auc_score(y, oof_xgb)
lgbm_cv = roc_auc_score(y, oof_lgbm)
cat_cv = roc_auc_score(y, oof_cat)

print(f"\n=== 20-Fold CV Results ===")
print(f"XGBoost CV AUC: {xgb_cv:.6f}")
print(f"LightGBM CV AUC: {lgbm_cv:.6f}")
print(f"CatBoost CV AUC: {cat_cv:.6f}")

## 7. Rank Blending & Final Submissions

In [ ]:
def rank_blend(arrays, weights):
    """Blend using rank-based method"""
    n = len(arrays[0])
    ranked = [rankdata(a) / n for a in arrays]
    blended = np.zeros(n)
    for w, r in zip(weights, ranked):
        blended += w * r
    return np.clip(blended, 0, 1)

# V21: XGBoost
submission_v21 = pd.DataFrame({
    'id': test_comp['id'],
    'Churn': test_xgb
})

# V22: LightGBM
submission_v22 = pd.DataFrame({
    'id': test_comp['id'],
    'Churn': test_lgbm
})

# V23: CatBoost
submission_v23 = pd.DataFrame({
    'id': test_comp['id'],
    'Churn': test_cat
})

# V24: Rank blend (0.40 XGB, 0.35 LGBM, 0.25 CAT)
v24_blend = rank_blend([test_xgb, test_lgbm, test_cat], [0.40, 0.35, 0.25])
submission_v24 = pd.DataFrame({
    'id': test_comp['id'],
    'Churn': v24_blend
})

# V25: Blend with pseudo-labels
v25_with_pseudo = rank_blend([test_xgb, test_lgbm, v24_blend], [0.35, 0.35, 0.30])
submission_v25 = pd.DataFrame({
    'id': test_comp['id'],
    'Churn': v25_with_pseudo
})

# Save submissions
submission_v21.to_csv('/kaggle/working/submission_v21_xgb_20fold.csv', index=False)
submission_v22.to_csv('/kaggle/working/submission_v22_lgbm_20fold.csv', index=False)
submission_v23.to_csv('/kaggle/working/submission_v23_cat_20fold.csv', index=False)
submission_v24.to_csv('/kaggle/working/submission_v24_blend_20fold.csv', index=False)
submission_v25.to_csv('/kaggle/working/submission_v25_pseudo_blend.csv', index=False)

print("\nSubmissions saved:")
print(" - submission_v21_xgb_20fold.csv")
print(" - submission_v22_lgbm_20fold.csv")
print(" - submission_v23_cat_20fold.csv")
print(" - submission_v24_blend_20fold.csv")
print(" - submission_v25_pseudo_blend.csv")

print(f"\nV21 (XGB) - Mean: {test_xgb.mean():.6f}")
print(f"V22 (LGBM) - Mean: {test_lgbm.mean():.6f}")
print(f"V23 (CAT) - Mean: {test_cat.mean():.6f}")
print(f"V24 (Blend) - Mean: {v24_blend.mean():.6f}")
print(f"V25 (Pseudo) - Mean: {v25_with_pseudo.mean():.6f}")